# Poster2Movie — 02. Reseña: cartel → sinopsis

Versión **base**: Qwen3-VL-2B-Instruct sin entrenar, con una instrucción (detalles en `p2m/resena.py`).

**Entorno: GPU (T4).**

## Preparar el entorno

Monta tu Google Drive (ahí vive el dataset, `poster2movie_datos/`), clona el repo e instala lo necesario. Si el dataset te lo compartieron, agrega la carpeta `poster2movie_datos` a *Mi unidad* como acceso directo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"
# El dataset compartido es solo de lectura: equipo4 guarda modelos/resultados en su propia carpeta.
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_equipo4"
SALIDAS = os.environ.get("P2M_SALIDAS", os.environ["P2M_DATOS"])
REPO = "AndreaHortaHernandez/poster2movie"
if not os.path.isdir("/content/repo"):
    !git clone https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git pull
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
!git checkout equipo4-resena
!git pull

## 1. Probar con un cartel

In [ ]:
from google.colab import files
from PIL import Image
from p2m.resena import Resenador

resenador = Resenador()
for nombre in files.upload():
    print(resenador.generar(Image.open(nombre).convert('RGB')))

## 2. Evaluar en test (200 carteles, ~15-20 min)

ROUGE-L y BERTScore contra la sinopsis real. Con `--generos predichos` usa los géneros del clasificador en vez de los reales. Resultados en Drive: `poster2movie_datos/resultados/resena/` (incluye los textos generados).

In [ ]:
del resenador  # libera la GPU
!python -m p2m.evaluar_resena --cuantos 200 --nombre equipo4_prompt

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
pd.read_csv(SALIDAS + "/resultados/resena/equipo4_prompt_ejemplos.csv")[["titulo", "generada", "real", "BERTScore"]].head(10)

## 3. Entrenar con LoRA

Afina las capas de atención del modelo de lenguaje (la torre de visión y el resto del modelo quedan congelados) sobre pares cartel → `overview` del split **train**. Por default toma una muestra de `CUANTOS_TRAIN` películas (no hace falta el train completo para notar un cambio); súbele si el tiempo lo permite.

**Nota:** esta sección no se pudo probar con GPU real antes de subirla — la primera corrida en Colab puede necesitar ajustes (nombres de capas, memoria, tamaño de lote). Revisa la celda de diagnóstico antes de armar el `LoraConfig`.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForImageTextToText, AutoProcessor
from p2m.resena import MODELO_RESENA, construir_instruccion

procesador = AutoProcessor.from_pretrained(MODELO_RESENA)
# bfloat16 (no float16, aunque sea mas lento en la T4): mas estable para entrenar,
# evita NaNs en el gradiente sin necesitar un GradScaler.
modelo_base = AutoModelForImageTextToText.from_pretrained(MODELO_RESENA, dtype=torch.bfloat16).to("cuda")

# Diagnostico: nombres reales de las capas lineales de la PRIMERA capa decodificadora.
# Sirve para confirmar los target_modules de abajo y ver si el nombre de la torre de
# vision se puede distinguir del modelo de lenguaje (varia segun version de transformers).
for nombre, modulo in modelo_base.named_modules():
    if isinstance(modulo, torch.nn.Linear) and ".layers.0." in nombre:
        print(nombre)

In [ ]:
import os
from peft import PeftModel

# Si Colab se desconecta/reinicia a medio entrenamiento, al volver a correr esta celda
# retoma el adaptador guardado en Drive en vez de empezar de cero (se guarda cada 10
# pasos en la celda de "Entrenar").
RUTA_CHECKPOINT = f"{SALIDAS}/modelos/resena_lora_checkpoint"

config_lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # Si el diagnostico de arriba muestra las mismas capas tambien dentro de la torre de
    # vision ("visual"/"vision" en el nombre), descomenta para no tocarla (necesita
    # peft reciente):
    # exclude_modules=r".*vis.*",
)
if os.path.isdir(RUTA_CHECKPOINT):
    print(f"Retomando adaptador guardado en {RUTA_CHECKPOINT}")
    modelo = PeftModel.from_pretrained(modelo_base, RUTA_CHECKPOINT, is_trainable=True)
else:
    modelo = get_peft_model(modelo_base, config_lora)
modelo.print_trainable_parameters()

### Datos de entrenamiento

Una muestra del split **train** (no se toca test ni val). Se eligen ids primero y luego se filtra al recorrer `datos.peliculas(...)`, para no cargar todas las imágenes del train completo a memoria.

In [ ]:
import random
from p2m import datos

CUANTOS_TRAIN = 3000  # ~9s/cartel medido en T4: con una GPU mas rapida debería bajar bastante
SEMILLA = 123

metadata_train = datos.metadata("train")
ids_entrenamiento = set(
    metadata_train.sample(n=min(CUANTOS_TRAIN, len(metadata_train)), random_state=SEMILLA)["id"]
)
ejemplos = [
    f for f in datos.peliculas("train", ["id", "imagen", "overview", "generos"])
    if f["id"] in ids_entrenamiento
]
print(f"{len(ejemplos)} peliculas de entrenamiento")

In [ ]:
def preparar_ejemplo(cartel, generos, overview):
    """Arma un ejemplo de entrenamiento tokenizando la conversacion COMPLETA (prompt +
    la respuesta real) de una sola vez, para que pixel_values/image_grid_thw y el indice
    rope multimodal de Qwen3-VL queden consistentes con el largo final. El prompt se
    enmascara (-100) en labels usando su largo, tokenizado aparte.

    El processor no extiende mm_token_type_ids para el texto de la respuesta que pegamos
    (queda mas corto que input_ids): se rellena con 0 (texto) hasta igualar el largo
    (confirmado con un print de formas: mm_token_type_ids salia 98 mas corto que
    input_ids, justo el tamano del overview agregado)."""
    instruccion = construir_instruccion(list(generos))
    mensajes_prompt = [{"role": "user", "content": [
        {"type": "image", "image": cartel},
        {"type": "text", "text": instruccion},
    ]}]
    mensajes_completos = mensajes_prompt + [
        {"role": "assistant", "content": [{"type": "text", "text": overview.strip()}]}
    ]

    solo_prompt = procesador.apply_chat_template(
        mensajes_prompt, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    )
    largo_prompt = solo_prompt["input_ids"].shape[1]

    entrada = procesador.apply_chat_template(
        mensajes_completos, tokenize=True, return_dict=True, return_tensors="pt",
    )

    if "mm_token_type_ids" in entrada:
        faltan = entrada["input_ids"].shape[1] - entrada["mm_token_type_ids"].shape[1]
        if faltan > 0:
            relleno = entrada["mm_token_type_ids"].new_zeros((entrada["mm_token_type_ids"].shape[0], faltan))
            entrada["mm_token_type_ids"] = torch.cat([entrada["mm_token_type_ids"], relleno], dim=1)

    etiquetas = entrada["input_ids"].clone()
    etiquetas[:, :largo_prompt] = -100
    entrada["labels"] = etiquetas
    return {k: v.to("cuda") for k, v in entrada.items()}

### Entrenar

Lote efectivo de `ACUMULAR` carteles (gradiente acumulado, no batch real: cada cartel trae un número distinto de tokens de imagen). ~1-2 horas estimadas para 3000 ejemplos en una T4 — **esto es un estimado, nadie lo cronometró todavía**; si se ve muy lento, baja `CUANTOS_TRAIN`.

In [ ]:
from torch.optim import AdamW
import time

EPOCAS = 1
ACUMULAR = 8
TASA_APRENDIZAJE = 2e-4

optimizador = AdamW([p for p in modelo.parameters() if p.requires_grad], lr=TASA_APRENDIZAJE)
modelo.train()

random.shuffle(ejemplos)
inicio, paso = time.time(), 0
for i, f in enumerate(ejemplos, start=1):
    cartel = datos.abrir_cartel(f["imagen"])
    lote = preparar_ejemplo(cartel, f["generos"], f["overview"])
    salida = modelo(**lote)
    (salida.loss / ACUMULAR).backward()
    if i % ACUMULAR == 0:
        optimizador.step()
        optimizador.zero_grad()
        paso += 1
        if paso % 5 == 0:
            print(f"paso {paso} ({i}/{len(ejemplos)}): loss {salida.loss.item():.3f} "
                  f"({time.time()-inicio:.0f}s)", flush=True)
        if paso % 10 == 0:
            modelo.save_pretrained(RUTA_CHECKPOINT)  # por si Colab se desconecta

## 4. Guardar y subir a Hugging Face

Se fusiona el adaptador LoRA con el modelo base (`merge_and_unload`) y se sube el modelo completo a Hugging Face Hub — así `evaluar_resena.py --modelo <repo>` lo carga igual que el base, sin tocar la interfaz de `Resenador`. Mismo flujo que con el traductor.

In [ ]:
modelo_final = modelo.merge_and_unload()
RUTA_MODELO = "/content/poster2movie_resena_lora"
modelo_final.save_pretrained(RUTA_MODELO)
procesador.save_pretrained(RUTA_MODELO)

In [ ]:
from huggingface_hub import login
login()  # pega un token de Hugging Face con permiso de escritura

REPO_HF = "tu-usuario-hf/poster2movie-resena-equipo4"  # cambia "tu-usuario-hf" por tu usuario real de HF
modelo_final.push_to_hub(REPO_HF)
procesador.push_to_hub(REPO_HF)

## 5. Evaluar la versión con LoRA

Libera la GPU del entrenamiento antes de evaluar (el evaluador carga su propia copia del modelo, desde el repo de Hugging Face que acabas de subir).

In [ ]:
del modelo, modelo_final, modelo_base, procesador
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!python -m p2m.evaluar_resena --cuantos 200 --nombre equipo4_lora --modelo tu-usuario-hf/poster2movie-resena-equipo4

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
print(pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_metricas.csv").to_string(index=False))
pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_ejemplos.csv")[["titulo", "generada", "real", "BERTScore"]].head(10)

## 6. Intento de arreglar el largo en la generación (descartado)

`equipo4_lora` salió con sinopsis cortas (~30 palabras contra ~58 reales). Se probaron dos
ajustes de generación sobre el modelo ya entrenado (sin reentrenar), ninguno sirvió:

| Prueba | BERTScore | Palabras | Qué pasó |
|---|---|---|---|
| `equipo4_lora` (original) | **0.154** | 29.9 | corto, pero fiel |
| + `min_new_tokens` | 0.061 | 72.9 | el modelo caía en **bucles de repetición** para rellenar |
| + `min_new_tokens` + anti-repetición | -0.044 | 89.3 | ya no repetía, pero **inventaba** datos/divagaba para no repetirse |

Conclusión: el problema es del entrenamiento, no de la generación — obligar al modelo a seguir
más allá de donde "quiere" parar siempre lo empeora. Se revirtieron estos cambios en
`p2m/resena.py`; `equipo4_lora` (versión original, arriba) queda como el resultado del rol C. Un
segundo entrenamiento (más épocas) sería la vía correcta para corregir el largo, pendiente como
trabajo futuro.

## 7. Segundo intento: generar en 2 pasos (borrador + expandir) — también descartado

En vez de forzar la decodificación (sección 6), se probó pedirle al modelo una segunda vez que
**expandiera su propio borrador** a 45-60 palabras. Tampoco funcionó: salió **igual de corto**
(26.7 palabras) y con BERTScore similar/menor (0.145 vs 0.154 del original).

**Conclusión real:** la instrucción de "2 pasos" le pedía el mismo largo (45-60 palabras) que la
instrucción original — y el LoRA ya demostró que ignora esa parte del prompt. El modelo con LoRA
aprendió a ser breve **en los pesos**, no por cómo se le pide; pedírselo de nuevo con otras
palabras no cambia nada. Con esto, los 3 intentos de arreglar el largo en la *generación* fallaron
(bucles, invención, y ahora simple indiferencia a la instrucción). Se revirtió `p2m/resena.py` a
la versión de una sola pasada. La solución tiene que ser en el *entrenamiento* — ver sección 8.

## 8. Diagnóstico antes de reentrenar

Un compañero de otro equipo revisó nuestros resultados y propuso una explicación mejor para el largo corto de `equipo4_lora` (~30 palabras contra ~58 reales): las sinopsis reales varían mucho (muchas tienen 35 palabras o menos), así que el modelo aprendió que puede cortar después de 1-2 oraciones. Por eso `min_new_tokens` lo hacía repetirse (sección 6) y pedirle el largo en el prompt no sirve (sección 7): en el entrenamiento aprendió a ignorar esa parte.

Antes de gastar otra ronda de entrenamiento completa, dos chequeos baratos:

1. Confirmar la distribución de largos en el train (sin GPU).
2. Probar beam search con `length_penalty` sobre el modelo YA entrenado, sin reentrenar — medido en **val** (no test, para no gastar la muestra oficial) y con más carteles que las pruebas descartadas (que usaron solo 30).

In [ ]:
from p2m import datos

metadata_train = datos.metadata("train")
largos_train = metadata_train["overview"].str.split().str.len()
print(largos_train.describe())
print(f"\n% con menos de 45 palabras: {(largos_train < 45).mean() * 100:.1f}%")
print(f"% entre 35 y 75 palabras:   {largos_train.between(35, 75).mean() * 100:.1f}%")

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from p2m import datos
from p2m.resena import construir_instruccion
from p2m.evaluar_resena import muestra_test

MODELO_HF = "Ddarwin08/poster2movie-resena-equipo4"
procesador_bs = AutoProcessor.from_pretrained(MODELO_HF)
modelo_bs = AutoModelForImageTextToText.from_pretrained(MODELO_HF, dtype=torch.float16).to("cuda").eval()


@torch.inference_mode()
def generar_beam(cartel, generos, length_penalty=1.5, num_beams=4):
    mensajes = [{"role": "user", "content": [
        {"type": "image", "image": cartel},
        {"type": "text", "text": construir_instruccion(list(generos))},
    ]}]
    entrada = procesador_bs.apply_chat_template(
        mensajes, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to("cuda")
    salida = modelo_bs.generate(
        **entrada, max_new_tokens=120, num_beams=num_beams, length_penalty=length_penalty, do_sample=False,
    )
    largo = entrada["input_ids"].shape[1]
    return procesador_bs.batch_decode(salida[:, largo:], skip_special_tokens=True)[0].strip()


# Muestra de VAL (no test): es para decidir si vale la pena, no para la metrica oficial.
muestra_val = muestra_test(20, split="val")
for f in muestra_val:
    cartel = datos.abrir_cartel(f["imagen"])
    generada = generar_beam(cartel, f["generos"])
    print(f"{f['title']}: {len(generada.split())} palabras (real: {len(f['overview'].split())})")
    print(generada)
    print()

In [ ]:
del modelo_bs, procesador_bs
import gc
gc.collect()
torch.cuda.empty_cache()

### Validar el beam search con metricas reales (descartado)

La prueba rapida de arriba (20 carteles, a ojo) se veia bien: ~44 palabras en vez de ~30, sin repeticion. Pero la metrica formal en val (60 carteles) dice otra cosa:

| | ROUGE-L | BERTScore | palabras |
|---|---|---|---|
| `equipo4_lora` (greedy, test oficial) | 0.145 | **0.154** | 29.9 / 58.3 |
| `equipo4_lora_beam` (beam, val, 60) | 0.151 | **0.129** | 48.9 / 61.1 |

El largo y el ROUGE-L mejoran un poco, pero el **BERTScore cae ~16%**. Viendo los ejemplos: con mas espacio que llenar, el modelo inventa detalles de trama especificos que no son los reales (*The Ninth Gate*: 0.085, *8 Million Ways to Die*: -0.049) — el mismo patron que la seccion 7 (generar en 2 pasos), solo que el mecanismo esta vez es beam search en vez de expandir. Con esto son 3 de 3 intentos de arreglar el largo en la *generacion* que fallan igual: el modelo no sabe la trama real, y darle mas margen para hablar (forzando tokens, expandiendo, o premiando secuencias largas con beam search) solo le da mas espacio para inventar. **Se revirtio el cambio en `p2m/resena.py`** (confirmado identico a antes del experimento). El fix tiene que venir del entrenamiento (seccion 9), no de como se decodifica.

In [ ]:
!python -m p2m.evaluar_resena --cuantos 60 --nombre equipo4_lora_beam --modelo Ddarwin08/poster2movie-resena-equipo4 --split val

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
print(pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_beam_metricas.csv").to_string(index=False))
pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_beam_ejemplos.csv")[["titulo", "generada", "real", "BERTScore"]].head(10)

## 9. Ronda 2 (revisada): reentrenar filtrando el largo

Sigue sin reentrenar desde cero: parte de `Ddarwin08/poster2movie-resena-equipo4` (ya fusionado) y le aplica un adaptador LoRA nuevo, con `TASA_APRENDIZAJE_2 = 5e-5` (más baja que la ronda 1, que se sospecha fue agresiva). Lo que cambia frente al intento anterior de esta sección:

- **Filtra los datos de entrenamiento** a sinopsis de 35 a 75 palabras (cerca de las 45-60 que pide la instrucción), para dejar de enseñarle el atajo de cortar corto.
- **El checkpoint ahora retoma de verdad**: antes, si Colab se desconectaba, al volver a correr la celda de entrenamiento se barajaba `ejemplos_2` de nuevo (sin semilla) y el contador de pasos volvía a 0 — el adaptador seguía desde donde quedó, pero no era 1 época limpia. Ahora el orden queda fijo (`random.Random(SEMILLA_2)`) y se guarda/retoma también el índice, el paso y el estado del optimizador.

**Corre primero con `CUANTOS_TRAIN_2` chico (ya está en 100)** para confirmar que el filtro y el checkpoint nuevo no truenan, antes de subirle.

**Necesitas correr también la celda de `preparar_ejemplo` de la sección 3** — es la misma función, no se duplica aquí.

In [ ]:
import os
import torch
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor

MODELO_RONDA1 = "Ddarwin08/poster2movie-resena-equipo4"  # ya entrenado y fusionado (ronda 1)

procesador = AutoProcessor.from_pretrained(MODELO_RONDA1)
modelo_base = AutoModelForImageTextToText.from_pretrained(MODELO_RONDA1, dtype=torch.bfloat16).to("cuda")

RUTA_CHECKPOINT_2 = f"{SALIDAS}/modelos/resena_lora_checkpoint_v2"
RUTA_ESTADO_2 = f"{RUTA_CHECKPOINT_2}_estado.json"  # indice, paso y optimizador para retomar de verdad

config_lora_2 = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
if os.path.isdir(RUTA_CHECKPOINT_2):
    print(f"Retomando adaptador de la ronda 2 guardado en {RUTA_CHECKPOINT_2}")
    modelo = PeftModel.from_pretrained(modelo_base, RUTA_CHECKPOINT_2, is_trainable=True)
else:
    modelo = get_peft_model(modelo_base, config_lora_2)
modelo.print_trainable_parameters()

### Datos de entrenamiento (ronda 2)

Filtra a sinopsis de `LARGO_MIN` a `LARGO_MAX` palabras antes de muestrear, para que el train ya no tenga tantos ejemplos cortos que le enseñen el atajo.

In [ ]:
import random
from p2m import datos

CUANTOS_TRAIN_2 = 1500  # (se probo primero con 100 para confirmar que no truena)
SEMILLA_2 = 456  # distinta de la ronda 1, para ver ejemplos nuevos
LARGO_MIN, LARGO_MAX = 35, 75  # cerca del rango que pide la instruccion (45-60), con margen

metadata_train_2 = datos.metadata("train")
largos_2 = metadata_train_2["overview"].str.split().str.len()
candidatos_2 = metadata_train_2[largos_2.between(LARGO_MIN, LARGO_MAX)]
print(f"{len(candidatos_2)} de {len(metadata_train_2)} peliculas de train caen en "
      f"[{LARGO_MIN}, {LARGO_MAX}] palabras")

ids_entrenamiento_2 = set(
    candidatos_2.sample(n=min(CUANTOS_TRAIN_2, len(candidatos_2)), random_state=SEMILLA_2)["id"]
)
ejemplos_2 = [
    f for f in datos.peliculas("train", ["id", "imagen", "overview", "generos"])
    if f["id"] in ids_entrenamiento_2
]
print(f"{len(ejemplos_2)} peliculas de entrenamiento (ronda 2, filtradas por largo)")

### Entrenar (ronda 2)

Usa `preparar_ejemplo` definida en la sección 3 (corre esa celda si no la has corrido en esta sesión). Orden fijo por semilla y checkpoint resumible: guarda el adaptador, el optimizador y el índice/paso cada 10 pasos en `RUTA_CHECKPOINT_2` (separado del de la ronda 1); si Colab se desconecta, retoma exactamente donde se quedó en vez de volver a barajar desde el principio.

In [ ]:
from torch.optim import AdamW
import json
import time

EPOCAS_2 = 1
ACUMULAR_2 = 8
TASA_APRENDIZAJE_2 = 5e-5  # mas baja que la ronda 1 (2e-4): se sospecha que fue muy agresiva

optimizador_2 = AdamW([p for p in modelo.parameters() if p.requires_grad], lr=TASA_APRENDIZAJE_2)

# Orden fijo (no al azar en cada sesion): asi se puede retomar exactamente en el mismo indice.
random.Random(SEMILLA_2).shuffle(ejemplos_2)

indice_inicial, paso = 0, 0
if os.path.isfile(RUTA_ESTADO_2):
    with open(RUTA_ESTADO_2) as f_estado:
        estado = json.load(f_estado)
    indice_inicial, paso = estado["indice"], estado["paso"]
    optimizador_2.load_state_dict(torch.load(f"{RUTA_CHECKPOINT_2}_optimizador.pt"))
    print(f"Retomando desde el ejemplo {indice_inicial} (paso {paso})")

modelo.train()
inicio = time.time()
for i, f in enumerate(ejemplos_2[indice_inicial:], start=indice_inicial + 1):
    cartel = datos.abrir_cartel(f["imagen"])
    lote = preparar_ejemplo(cartel, f["generos"], f["overview"])
    salida = modelo(**lote)
    (salida.loss / ACUMULAR_2).backward()
    if i % ACUMULAR_2 == 0:
        optimizador_2.step()
        optimizador_2.zero_grad()
        paso += 1
        if paso % 5 == 0:
            print(f"paso {paso} ({i}/{len(ejemplos_2)}): loss {salida.loss.item():.3f} "
                  f"({time.time()-inicio:.0f}s)", flush=True)
        if paso % 10 == 0:
            modelo.save_pretrained(RUTA_CHECKPOINT_2)
            torch.save(optimizador_2.state_dict(), f"{RUTA_CHECKPOINT_2}_optimizador.pt")
            with open(RUTA_ESTADO_2, "w") as f_estado:
                json.dump({"indice": i, "paso": paso}, f_estado)
print(f"Terminado: {paso} pasos en total")

## 10. Guardar, subir a HF y evaluar la ronda 2

Mismo flujo que la ronda 1, con nombres distintos (`-v2`) para no pisar el modelo ni los resultados que ya tenemos. Primero se evalúa en **val** con una muestra más grande que las pruebas descartadas (60, no 30) — si se ve bien ahí, se sube `CUANTOS_TRAIN_2` y se corre la evaluación oficial de **test** (200, sin `--split`) con el nombre final `equipo4_lora_v2`.

In [ ]:
modelo_final_2 = modelo.merge_and_unload()
RUTA_MODELO_2 = "/content/poster2movie_resena_lora_v2"
modelo_final_2.save_pretrained(RUTA_MODELO_2)
procesador.save_pretrained(RUTA_MODELO_2)

In [ ]:
from huggingface_hub import login
login()  # pega un token de Hugging Face con permiso de escritura

REPO_HF_2 = "tu-usuario-hf/poster2movie-resena-equipo4-v2"  # cambia "tu-usuario-hf"
modelo_final_2.push_to_hub(REPO_HF_2)
procesador.push_to_hub(REPO_HF_2)

In [ ]:
del modelo, modelo_final_2, modelo_base, procesador, optimizador_2
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!python -m p2m.evaluar_resena --cuantos 60 --nombre equipo4_lora_v2_prueba --modelo tu-usuario-hf/poster2movie-resena-equipo4-v2 --split val

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
print(pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_v2_prueba_metricas.csv").to_string(index=False))
pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_v2_prueba_ejemplos.csv")[["titulo", "generada", "real", "BERTScore"]].head(10)

### Resultado de la ronda 2 (descartada)

Con los 1500 ejemplos filtrados (35-75 palabras) y el checkpoint ya arreglado (resumible de
verdad), el entrenamiento termino sin problemas: 187 pasos, sobrevivio a dos desconexiones de
Colab a medio camino. Pero el resultado en val (60 carteles) es el peor de los tres intentos:

| | ROUGE-L | BERTScore | palabras |
|---|---|---|---|
| `equipo4_lora` (ronda 1, greedy, **test** oficial) | 0.145 | **0.154** | 29.9 / 58.3 |
| `equipo4_lora_beam` (beam, val, 60) | 0.151 | 0.129 | 48.9 / 61.1 |
| `equipo4_lora_v2_prueba` (ronda 2, val, 60) | 0.144 | **0.119** | 45.1 / 61.0 |

El largo si mejoro (45 palabras, mas cerca del real de 61), pero viendo los ejemplos generados el
problema de fondo no se arreglo: el modelo cae en **repeticion degenerada** para rellenar
palabras (ej. "...forced to work in the camp's laundry..." repetido 9 veces seguidas en un solo
ejemplo), y el contenido sigue siendo generico/inventado, sin conectar con la trama real del
cartel. Mas palabras, pero no mas significado -- por eso el BERTScore es el peor de los tres.

Con esto son 4 de 4 intentos de arreglar el largo (forzar `min_new_tokens`, generar en dos
pasadas, beam search, reentrenar con datos filtrados por largo) que empeoran el BERTScore
respecto al modelo original. **Se abandona la ronda 2.**

El modulo final se queda con el modelo de la ronda 1 (`Ddarwin08/poster2movie-resena-equipo4`),
sin cambios en la generacion (`p2m/resena.py` ya apunta ahi via `MODELO_RESENA`). Mejora el
zero-shot original en ROUGE-L/BERTScore aunque sus sinopsis sean mas cortas que las reales; los 4
intentos de alargarlas empeoraron mas de lo que arreglaron.